# AI Newsroom Studio — Pipeline Notebook
10-agent pipeline: HackerNews trends → fact-check → script → video → YouTube Shorts.

**Agents built:** Agent 1 (Trend Hunter) · Agent 2 (Context Researcher) · Agent 3 (Fact Checker)

---
## Setup: Shared State

In [1]:
from typing_extensions import TypedDict

class NewsroomState(TypedDict):
    stories: dict   # keyed by story_id (slug), enriched in place by each agent

---
## Agent 1: Trend Fetcher
Fetches top HackerNews stories and scores them by velocity.

**Velocity** = (upvotes + comments×2) / age_hrs — rewards recent engagement.

In [2]:
from agents.agent1 import fetch_trends
import re

def make_story_id(title: str) -> str:
    """Slugify title to a stable dict key. e.g. 'Melody India Italy' → 'melody-india-italy'"""
    return re.sub(r'[^a-z0-9]+', '-', title.lower()).strip('-')

# AGENT 1 NODE
def trend_hunter_node(state: NewsroomState) -> dict:
    trends = fetch_trends()
    stories = {make_story_id(t["title"]): t for t in trends}
    return {"stories": stories}

In [3]:
# Run Agent 1
call = trend_hunter_node(NewsroomState(stories={}))
print(f"Stories fetched: {len(call['stories'])}")
for sid, s in call['stories'].items():
    print(f"  {s['velocity']:>6.1f} vel | {sid}")

Stories fetched: 8
   157.2 vel | age-verification-is-just-a-precursor-to-automated-attribution-of-speech
   125.5 vel | pollen-ceo-negus-fancey-cto-wright-tried-to-remove-article-and-google-helped
    98.3 vel | glm-5-2-beats-claude-in-our-benchmarks
    97.3 vel | hackerrank-open-sourced-its-ats-my-resume-scored-90-100-oh-wait-74-no-88
    33.7 vel | historical-memory-prices-1960-2026
     5.6 vel | dissecting-apple-s-sparse-image-format-asif
     1.0 vel | we-found-a-bug-in-the-hyper-http-library
     0.3 vel | numa-cores-memory-and-the-distance-between-them


---
## Agent 2: Context Researcher
For each story:
1. Fetches article content (3-tier: trafilatura → Jina → Tavily)
2. Gathers background (DDG news + Wikipedia)
3. Synthesises one tight background paragraph (qwen2.5:3b, content-anchored)

**Key design:**  rejects page chrome before accepting fetched text.
Backgrounds are honest-empty when DDG has no coverage (GitHub/arXiv/new tools — see KNOWN_ISSUES.md).

In [4]:
import ollama, subprocess, time

# Health-check: Ollama must be running for Agent 2 synthesis
try:
    ollama.list()
    print("✅ Ollama already running")
except Exception:
    try:
        subprocess.Popen(["ollama", "serve"])
        time.sleep(3)
        ollama.list()
        print("✅ Ollama started")
    except Exception:
        print("✗ Run  in a terminal manually")

✅ Ollama already running


In [5]:
from agents.agent2 import fetch_url_content, fetch_trend_background

# AGENT 2 NODE
def context_researcher_node(state: NewsroomState) -> dict:
    """Enriches each story in-place with content + background keys."""
    stories = state["stories"]
    for sid, story in stories.items():
        print(f"Agent 2 Starts For : {sid}")
        story["content"]    = fetch_url_content(story["url"])
        story["background"] = fetch_trend_background(story["title"], story["content"])
        print(f"  Agent 2 Ends for: {story['title']}")
        print("=" * 80)
    return {"stories": stories}

In [6]:
# Run Agent 2 (enriches call['stories'] in-place)
call2 = context_researcher_node(call)

Agent 2 Starts For : age-verification-is-just-a-precursor-to-automated-attribution-of-speech
Trafiltura Success ✅  In Loading Content :2509 characters
  [background] gathering snippets for: Age verification is just a precursor to automated attribution of speech
[ddg] error for 'Age verification is just a precursor to automated attribution of speech': No results found.
  [bg] DDG returned 0 snippets
  [bg] Wikipedia search returns: 2020 chars
  [background] 1 snippets, synthesizing (content-anchored)...
  [synth] 1 snippets, 2107 snippet chars → llama3.1:8b
  [synth] llama3.1:8b → 665 chars
Result After Background Syntezing is : In recent years, numerous countries including the US, European nations, and Australia have introduced "age verification" regulations to restrict access to certain online content. These laws are often presented as a means of protecting children from inappropriate material, but they also enable governments to attribute digital identities to physical ones, effectiv

/Users/deepakrathore/Documents/Agentic-AI/Examples/NewsStudio/multi-agent-env/lib/python3.13/site-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /Users/deepakrathore/Documents/Agentic-AI/Examples/NewsStudio/multi-agent-env/lib/python3.13/site-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


  [bg] Wikipedia search returns: empty (skipped)
  [background] 3 snippets, synthesizing (content-anchored)...
  [synth] 3 snippets, 899 snippet chars → llama3.1:8b
  [synth] llama3.1:8b → 489 chars
Result After Background Syntezing is : Apple introduced a new disk image format called Apple Sparse Image Format (ASIF) in macOS 26 Tahoe at WWDC 2025. ASIF is designed for use with virtual machines and takes inspiration from existing virtual disk formats, making it similar to sparse VMDK, VHDX, or QCOW2 files. The format allows for storing large disks or files in a smaller "sparse" manner. To understand the file format, the author created a test file using Apple's documentation and wrote a parser to dissect its structure.
  [bg] background: 489 chars
  Agent 2 Ends for: Dissecting Apple's Sparse Image Format (ASIF)
Agent 2 Starts For : we-found-a-bug-in-the-hyper-http-library
Trafiltura Loaded Junk Content Discarding & Switching To Jina
Jina Loaded Junk Content Discarding & Switching To Ta

In [7]:
# Inspect Agent 2 output
for sid, story in call2["stories"].items():
    content_len  = len(story.get("content",  ""))
    bg_len       = len(story.get("background",""))
    bg_status    = f"{bg_len} chars" if bg_len else "EMPTY"
    print(f"{story['velocity']:>6.1f} vel | content {content_len:>5}c | bg {bg_status:<12} | {story['title'][:55]}")

 157.2 vel | content  2509c | bg 665 chars    | Age verification is just a precursor to automated attri
 125.5 vel | content  5582c | bg 685 chars    | Pollen (CEO Negus-Fancey, CTO Wright) tried to remove a
  98.3 vel | content  6000c | bg 700 chars    | GLM 5.2 beats Claude in our benchmarks
  97.3 vel | content  5417c | bg 475 chars    | HackerRank open sourced its ATS. My resume scored 90/10
  33.7 vel | content  3966c | bg 568 chars    | Historical memory prices 1960-2026
   5.6 vel | content  6000c | bg 489 chars    | Dissecting Apple's Sparse Image Format (ASIF)
   1.0 vel | content     0c | bg EMPTY        | We found a bug in the hyper HTTP library
   0.3 vel | content  6000c | bg 770 chars    | NUMA: Cores, memory, and the distance between them


---
## Agent 2: Tests

**Manual tests** — run these at different times of day to stress-test the pipeline.

**Automated tests** — pure-function unit tests, run before every commit.

In [8]:
# AUTOMATED UNIT TESTS (deterministic pure-function checks)
from agents.agent2 import looks_like_real_content, clean_jina

def test_rejects_language_menu():
    menu = "sign in\nlanguage\n简体中文\n日本語\n한국어"
    assert looks_like_real_content(menu) == False, "should reject a language menu"

def test_accepts_real_article():
    article = (
        "Valve announced Steam Machine pricing today after months of delays.\n"
        "The base model starts at $1,049 with component costs driving the price.\n"
        "Pre-orders open June 8th ahead of the June 29th release date.\n"
        "The semi-custom AMD platform faced supply chain challenges this year."
    )
    assert looks_like_real_content(article) == True, "should accept a real multi-line article"

def test_clean_jina_strips_header():
    text = "Title: X\n\nMarkdown Content:\n" + "real body " * 50
    assert not clean_jina(text).startswith("Title:"), "should strip Jina header"

# Run them
test_rejects_language_menu()
test_accepts_real_article()
test_clean_jina_strips_header()
print("✅ all tests passed")

✅ all tests passed


---
## Agent 3: Fact Checker
Scores each story's credibility using two signals:

| Signal | Weight | How |
|--------|--------|-----|
|  | 25% | Domain trust map (github/arxiv/reuters etc.) |
|  | 75% | Groq gpt-oss-120b classifies: REAL / OPINION / SPAM |

**Why a 120B cloud model?**
The local 3B model judged credibility from *writing tone* alone — it wrongly labelled
real products (Haystack, Bunny DNS, Krea 2) as WEAK because they sounded promotional.
A 120B model has broad world knowledge to recognise these as genuine entities.

**Key design decisions:**
- REAL/OPINION/SPAM categories (not a decimal) — 3B+ models classify reliably but rate decimals poorly
- Soft score only — story is *marked* discarded, not deleted (audit trail; Agent 4 can override)
- Groq failure → 0.5 neutral (never discard due to an API crash)
- Thin content < 500 chars → 0.5 neutral (too little to judge reliably)

In [9]:
from agents.agent3 import source_score, llm_credibility_check, check_credibility
print("✅ Agent 3 imported")

✅ Agent 3 imported


In [10]:
# AGENT 3 NODE
def fact_checker_node(state: NewsroomState) -> dict:
    """Enriches each story in-place with credibility_score + discarded keys."""
    stories = state["stories"]
    for sid, story in stories.items():
        check_credibility(story)
        flag = "🗑️ DISCARD" if story["discarded"] else "✅ KEEP"
        print(f"{story['credibility_score']:.2f} {flag} | {story['title'][:55]}")
        time.sleep(1)
    return {"stories": stories}

In [11]:
# ── FULL RUN: Agent 1 → 2 → 3 on real HN stories ──────────────────────
# RESTART KERNEL before this cell if you edited agent3.py
# (Jupyter caches imports — stale agent3 = wrong results)

# Step 1: fresh trends
fresh_call  = trend_hunter_node(NewsroomState(stories={}))
# Step 2: enrich with content + background
fresh_call2 = context_researcher_node(fresh_call)
# Step 3: score credibility on REAL content (thousands of chars → guard passes)
print("── CREDIBILITY RESULTS ─────────────────────────────")
fresh_call3 = fact_checker_node(fresh_call2)
print("────────────────────────────────────────────────────")

Agent 2 Starts For : age-verification-is-just-a-precursor-to-automated-attribution-of-speech
Trafiltura Success ✅  In Loading Content :2509 characters
  [background] gathering snippets for: Age verification is just a precursor to automated attribution of speech
[ddg] error for 'Age verification is just a precursor to automated attribution of speech': No results found.
  [bg] DDG returned 0 snippets
  [bg] Wikipedia search returns: 2020 chars
  [background] 1 snippets, synthesizing (content-anchored)...
  [synth] 1 snippets, 2107 snippet chars → llama3.1:8b
  [synth] llama3.1:8b → 826 chars
Result After Background Syntezing is : In recent years, numerous countries including the US, European nations, and Australia have introduced "age verification" regulations to restrict access to certain online content. These laws are often presented as a means of protecting children from online harm, but they also serve as a precursor to automated attribution of speech, allowing governments to quickly

In [12]:
# ── INSPECT: full story details after all 3 agents ─────────────────────
for sid, story in fresh_call3["stories"].items():
    flag = "🗑️" if story.get("discarded") else "✅"
    print(f"{flag} {story['credibility_score']:.2f} | vel {story['velocity']:>6.1f} | "
          f"content {len(story.get('content','')):>5}c | "
          f"bg {len(story.get('background','')):>4}c | {story['title'][:50]}")

✅ 0.50 | vel  156.3 | content  2509c | bg  826c | Age verification is just a precursor to automated 
✅ 0.50 | vel  129.4 | content  5582c | bg  684c | Pollen (CEO Negus-Fancey, CTO Wright) tried to rem
✅ 0.80 | vel   98.1 | content  6000c | bg  631c | GLM 5.2 beats Claude in our benchmarks
✅ 0.50 | vel   97.3 | content  5417c | bg  548c | HackerRank open sourced its ATS. My resume scored 
✅ 0.80 | vel   33.7 | content  3966c | bg  657c | Historical memory prices 1960-2026
✅ 0.80 | vel    5.9 | content  6000c | bg  481c | Dissecting Apple's Sparse Image Format (ASIF)
🗑️ 0.35 | vel    1.0 | content     0c | bg    0c | We found a bug in the hyper HTTP library
✅ 0.80 | vel    0.3 | content  6000c | bg  822c | NUMA: Cores, memory, and the distance between them
